In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

In [4]:
delhi_df = pd.read_csv("price_sqft_delhi.csv")
gurgaon_df = pd.read_csv("price_sqft_gurgaon.csv")
mumbai_df = pd.read_csv("price_sqft_mumbai.csv")

/tmp/ipykernel_326/1964963143.py:2: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  gurgaon_df = pd.read_csv("price_sqft_gurgaon.csv")


In [5]:
print("Delhi dataset shape:", delhi_df.shape)
print("Gurgaon dataset shape:", gurgaon_df.shape)
print("Mumbai dataset shape:", mumbai_df.shape)

Delhi dataset shape: (7738, 18)
Gurgaon dataset shape: (10704, 67)
Mumbai dataset shape: (9514, 55)


In [6]:
delhi_df["city"] = "Delhi"
gurgaon_df["city"] = "Gurgaon"
mumbai_df["city"] = "Mumbai"

In [7]:
print("Delhi columns:\n", delhi_df.columns.tolist(), "\n")
print("Gurgaon columns:\n", gurgaon_df.columns.tolist(), "\n")
print("Mumbai columns:\n", mumbai_df.columns.tolist(), "\n")

Delhi columns:
 ['Unnamed: 0', 'price', 'Address', 'area', 'latitude', 'longitude', 'Bedrooms', 'Bathrooms', 'Balcony', 'Status', 'neworold', 'parking', 'Furnished_status', 'Lift', 'Landmarks', 'type_of_building', 'desc', 'Price_sqft', 'city'] 

Gurgaon columns:
 ['PROP_ID', 'PHOTO_URL', 'MEDIUM_PHOTO_URL', 'PREFERENCE', 'DESCRIPTION', 'PROPERTY_TYPE', 'CITY', 'LOCALITY', 'TRANSACT_TYPE', 'OWNTYPE', 'BEDROOM_NUM', 'BATHROOM_NUM', 'BALCONY_NUM', 'PRICE_PER_UNIT_AREA', 'FURNISH', 'FACING', 'AGE', 'FLOOR_NUM', 'TOTAL_FLOOR', 'FEATURES', 'REGISTER_DATE', 'PROP_NAME', 'MIN_PRICE', 'MAX_PRICE', 'PRICE_SQFT', 'LISTING', 'BUILDING_ID', 'CARPET_SQFT', 'SUPERBUILTUP_SQFT', 'BROKERAGE', 'MAP_DETAILS', 'FSL_Data', 'MIN_AREA_SQFT', 'MAX_AREA_SQFT', 'FORMATTED', 'AMENITIES', 'TOP_USPS', 'PD_URL', 'EXPIRY_DATE', 'GROUP_NAME', 'AREA', 'PRICE', 'PROP_HEADING', 'PROP_DETAILS_URL', 'CLASS_HEADING', 'CLASS_LABEL', 'SECONDARY_TAGS', 'PROPERTY_IMAGES', 'THUMBNAIL_IMAGES', 'TOTAL_LANDMARK_COUNT', 'FORMATTED_

In [8]:
# =========================================================
# COMPLETE REAL ESTATE DATA CLEANING PIPELINE (FIXED)
# =========================================================

import pandas as pd
import numpy as np
import re

# ---------- 1. DELHI CLEANING ----------

delhi = delhi_df.copy()

delhi = delhi.rename(columns={
    "Bedrooms":"bedrooms",
    "Bathrooms":"bathrooms",
    "Balcony":"balcony",
    "Furnished_status":"furnish",
    "type_of_building":"property_type"
})

delhi["price"] = pd.to_numeric(delhi["price"], errors="coerce")
delhi["area"] = pd.to_numeric(delhi["area"], errors="coerce")

delhi["price_sqft"] = delhi["price"] / delhi["area"]

delhi = delhi[
[
"price","area","bedrooms","bathrooms","balcony",
"price_sqft","latitude","longitude",
"furnish","property_type","city"
]
]

# ---------- 2. GURGAON CLEANING ----------

gurgaon = gurgaon_df.copy()

# extract lat long
gurgaon["latitude"] = gurgaon["MAP_DETAILS"].astype(str).str.extract(r"'LATITUDE':\s*'([\d\.]+)'")
gurgaon["longitude"] = gurgaon["MAP_DETAILS"].astype(str).str.extract(r"'LONGITUDE':\s*'([\d\.]+)'")

# correct columns
gurgaon["price"] = pd.to_numeric(gurgaon["MIN_PRICE"], errors="coerce")
gurgaon["area"] = pd.to_numeric(gurgaon["SUPERBUILTUP_SQFT"], errors="coerce")

# price per sqft already exists
gurgaon["price_sqft"] = pd.to_numeric(gurgaon["PRICE_SQFT"], errors="coerce")

gurgaon["bedrooms"] = np.nan
gurgaon["bathrooms"] = np.nan
gurgaon["balcony"] = np.nan

gurgaon["furnish"] = "Unknown"
gurgaon["property_type"] = "Apartment"
gurgaon["city"] = "Gurgaon"

gurgaon = gurgaon[
[
"price","area","bedrooms","bathrooms","balcony",
"price_sqft","latitude","longitude",
"furnish","property_type","city"
]
]
# ---------- 3. MUMBAI CLEANING ----------

import numpy as np

mumbai = mumbai_df.copy()

# extract latitude longitude
mumbai["latitude"] = mumbai["MAP_DETAILS"].astype(str).str.extract(r"'LATITUDE':\s*'([\d\.]+)'")
mumbai["longitude"] = mumbai["MAP_DETAILS"].astype(str).str.extract(r"'LONGITUDE':\s*'([\d\.]+)'")

# main features
mumbai["price"] = pd.to_numeric(mumbai["MIN_PRICE"], errors="coerce")
mumbai["area"] = pd.to_numeric(mumbai["MIN_AREA_SQFT"], errors="coerce")
mumbai["price_sqft"] = pd.to_numeric(mumbai["PRICE_PER_UNIT_AREA"], errors="coerce")

# missing features
mumbai["bedrooms"] = np.nan
mumbai["bathrooms"] = np.nan
mumbai["balcony"] = np.nan

mumbai["furnish"] = "Unknown"
mumbai["property_type"] = "Apartment"
mumbai["city"] = "Mumbai"

# keep final schema
mumbai = mumbai[
[
"price",
"area",
"bedrooms",
"bathrooms",
"balcony",
"price_sqft",
"latitude",
"longitude",
"furnish",
"property_type",
"city"
]
]


# ---------- FIX DUPLICATE COLUMN INDEXES ----------
delhi = delhi.loc[:, ~delhi.columns.duplicated()]
gurgaon = gurgaon.loc[:, ~gurgaon.columns.duplicated()]
mumbai = mumbai.loc[:, ~mumbai.columns.duplicated()]

# ---------- 4. MERGE DATASETS ----------
df = pd.concat([delhi, gurgaon, mumbai], ignore_index=True)

# Fix missing area using price and price_sqft
df["area"] = df["area"].fillna(df["price"] / df["price_sqft"])

print("Rows before cleaning:", df.shape)

print("\nMissing values:")
print(df[["price","area","price_sqft"]].isna().sum())

print("\nSample price values:")
print(df["price"].head(10))

print("\nSample area values:")
print(df["area"].head(10))

print("Merged shape:", df.shape)

# ---------- 5. BASIC CLEANING ----------
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

df = df.dropna(subset=["price","area"])
df["price_sqft"] = df["price_sqft"].fillna(df["price"] / df["area"])

df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].median())
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())
df["balcony"] = df["balcony"].fillna(0)

df["furnish"] = df["furnish"].fillna("Unknown")
df["property_type"] = df["property_type"].fillna("Unknown")

# ---------- 6. REMOVE OUTLIERS ----------
df = df[(df["area"] > 300) & (df["area"] < 20000)]
df = df[(df["price_sqft"] > 500) & (df["price_sqft"] < 500000)]

# ---------- 7. REMOVE DUPLICATES ----------
df = df.drop_duplicates()

# ---------- 8. ENCODE CATEGORICAL FEATURES ----------
df = pd.get_dummies(
    df,
    columns=["furnish","property_type","city"],
    drop_first=True
)

# ---------- 9. REMOVE TARGET LEAKAGE ----------
df = df.drop(columns=["price"], errors="ignore")

print("Final dataset shape:", df.shape)


Rows before cleaning: (27956, 11)

Missing values:
price         0
area          1
price_sqft    0
dtype: int64

Sample price values:
0     5600000.0
1     8800000.0
2    16500000.0
3     3810000.0
4     6200000.0
5     3700000.0
6     3270000.0
7     3990000.0
8     3500000.0
9     5500000.0
Name: price, dtype: float64

Sample area values:
0    1350.0
1    1490.0
2    2385.0
3    1050.0
4    1350.0
5    1150.0
6     890.0
7     850.0
8     960.0
9    1400.0
Name: area, dtype: float64
Merged shape: (27956, 11)
Final dataset shape: (16700, 14)


# MODEL PREPARATION NOW

In [9]:
X = df.drop(columns=["price_sqft"])
y = df["price_sqft"]

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [11]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_estimators=200, n_jobs=-1,
                      random_state=42)

In [12]:
y_pred = rf.predict(X_test)

In [13]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 16990.465043458367
R2 Score: 0.7005053838730246


In [14]:
df["price_sqft"].describe()


,price_sqft
count,16700.000000
mean,15332.351720
std,33540.454684
min,525.000000
25%,5000.000000
50%,7933.942149
75%,12500.000000
max,477941.000000


In [15]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


RandomForestRegressor(max_depth=30, min_samples_leaf=2, min_samples_split=5,
                      n_estimators=500, n_jobs=-1, random_state=42)

In [16]:
low = df["price_sqft"].quantile(0.01)
high = df["price_sqft"].quantile(0.99)

df = df[(df["price_sqft"] >= low) & (df["price_sqft"] <= high)]

In [17]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

,0
area,0.671052
longitude,0.163851
latitude,0.141187
city_Gurgaon,0.023078
furnish_Unknown,0.000322
bathrooms,0.000165
bedrooms,0.000098
property_type_Flat,0.000097
balcony,0.000068
furnish_Semi-Furnished,0.000027


In [18]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=20, random_state=42)
df["location_cluster"] = kmeans.fit_predict(df[["latitude","longitude"]])

In [19]:
print("Train R2:", rf.score(X_train, y_train))
print("Test R2:", rf.score(X_test, y_test))

Train R2: 0.8867775232603232
Test R2: 0.7094298416329519


In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 16735.406326054035
MAE: 4942.707812687158
R2: 0.7094298416329519


In [21]:
import joblib

joblib.dump(rf, "price_model.pkl")

['price_model.pkl']